In [ ]:
# Kaggle notebook used https://www.kaggle.com/code/omeroruccelik/content-based-recommendation-systems

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.metrics.pairwise import euclidean_distances
import cudf

In [ ]:
# Extension to use GPU
%load_ext cudf.pandas

In [ ]:
nrows = 13000
skiprows = 0

header = pd.read_csv(r'movies.csv', delimiter=';', nrows=0).columns

batches = {}

for i in range(1, 8):
    data = pd.read_csv(r'movies.csv', delimiter=';', skiprows=skiprows + 1, nrows=nrows, names=header)
    print(f'Batch {i} - SKIPROWS: {skiprows}')

    batches[f'batch_{i}'] = data

    skiprows += nrows

Batch 1 - SKIPROWS: 0
Batch 2 - SKIPROWS: 13000
Batch 3 - SKIPROWS: 26000
Batch 4 - SKIPROWS: 39000
Batch 5 - SKIPROWS: 52000
Batch 6 - SKIPROWS: 65000
Batch 7 - SKIPROWS: 78000


In [ ]:
def preprocess_batch(data):
    data.dropna(subset=['actors', 'director', 'writer', 'original_title', 'description', 'genre'], inplace=True, axis=0)
    # data.dropna(subset=['actors', 'original_title', 'description', 'genre'], inplace=True, axis=0)
    # data.dropna(subset=['original_title', 'description'], inplace=True, axis=0)

    data = data.reset_index(drop=True)

    data['description'] = [re.sub(r'[^\w\s]', '', str(t)) for t in data['description']]
    data['actors'] = [re.sub(',', ' ', re.sub(' ', '', str(t))) for t in data['actors']]
    data['director'] = [re.sub(',', ' ', re.sub(' ', '', str(t))) for t in data['director']]
    data['writer'] = [re.sub(',', ' ', re.sub(' ', '', str(t))) for t in data['writer']]
    data['original_title'] = [re.sub(r'[^\w\s]', '', str(t)) for t in data['original_title']]
    data['original_title'] = [str(t).strip() for t in data['original_title']]
    data['genre'] = [re.sub(',', ' ', re.sub(' ', '', str(t))) for t in data['genre']]

    data["combined"] = (
        data['genre'] + '  ' + data['actors'] + ' ' + data['director'] + ' '
        + data['writer'] + ' ' + data['original_title'] + ' ' + data['description']
    )
    # data["combined"] = (data['original_title'] + ' ' + data['description'])
    # data["combined"] = (
    #     data['genre'] + '  ' + data['actors'] + ' ' + data['original_title'] + ' ' + data['description']
    # )

    data.drop(['actors', 'director', 'writer', 'description', 'genre'], axis=1, inplace=True)
    # data.drop(['actors', 'description', 'genre'], axis=1, inplace=True)
    # data.drop(['description'], axis=1, inplace=True)

    return data

In [ ]:
%%time

preprocessed_batches = {}
tfidf_batches = {}
cosine_similarities_batches = {}
euclidean_similarities_batches = {}
movie_titles_batches = {}
indices_batches = {}

# Vectorizer instance for TF-IDF transformation
vectorizer = TfidfVectorizer()

# Process each batch: preprocess, vectorize, and calculate cosine similarity
for i in range(1, 8):
    print(f'Processing Batch {i}')

    # Preprocess the raw batch
    raw_batch = batches[f'batch_{i}']
    preprocessed_batch = preprocess_batch(raw_batch)

    # Store preprocessed batch in dictionary
    preprocessed_batches[f'batch_{i}'] = preprocessed_batch

    # Apply TF-IDF vectorization on "combined" column
    matrix_batch = vectorizer.fit_transform(preprocessed_batch["combined"])

    # Store TF-IDF matrix in dictionary
    tfidf_batches[f'batch_{i}'] = matrix_batch

    # Calculate cosine similarity and euclidean distance matrix for the batch
    # cosine_similarities_batch = linear_kernel(matrix_batch, matrix_batch)
    euclidean_dist_batch = euclidean_distances(matrix_batch, matrix_batch)

    # Store cosine similarity and euclidean distance matrix in dictionary
    # cosine_similarities_batches[f'batch_{i}'] = cosine_similarities_batch
    euclidean_similarities_batches[f'batch_{i}'] = 1 / (1 + euclidean_dist_batch)

    # Extract movie titles and indices for the batch
    movie_title_batch = preprocessed_batch['original_title']
    indices_batch = pd.Series(preprocessed_batch.index, index=preprocessed_batch['original_title'])

    # Store movie titles and indices in dictionaries
    movie_titles_batches[f'batch_{i}'] = movie_title_batch
    indices_batches[f'batch_{i}'] = indices_batch

movie_to_global_index = pd.Series(data.index, index=data['original_title'])

Processing Batch 1
Processing Batch 2
Processing Batch 3
Processing Batch 4
Processing Batch 5
Processing Batch 6
Processing Batch 7
CPU times: user 42.1 s, sys: 16 s, total: 58.1 s
Wall time: 1min 13s


In [ ]:
# Function to sort the movies based on similarity across all batches
def content_recommender_all_batches(title):
    # Initialize a list to store similarity scores from all batches
    all_sim_scores = []

    # Iterate through all batches
    # for batch_id in cosine_similarities_batches.keys():
    for batch_id in euclidean_similarities_batches.keys():
        # Get indices, cosine similarities or euclidean similarities, and movie titles for the current batch
        indices = indices_batches[batch_id]
        # cosine_similarities = cosine_similarities_batches[batch_id]
        euclidean_similarities = euclidean_similarities_batches[batch_id]
        movie_title = movie_titles_batches[batch_id]

        # Check if the title exists in the current batch, searching in indices.index
        if title in indices.index or any(title in t for t in indices.index):
            # Find the index of the given title in this batch using .get to handle missing keys
            try:
                # Attempt to find the exact index
                idx = indices[title]
            except KeyError:
                # If not found, try to find a partial match using string contains
                try:
                    # Find matching indices using string contains
                    # matching_indices = indices.index[indices.index.str.startswith(title)]
                    matching_indices = indices.index[indices.index.str.contains(title)]
                    # Get the first matching index
                    idx = indices[matching_indices[0]]
                except IndexError:
                    # Handle case where no partial match is found
                    print(f"Title '{title}' not found in batch {batch_id}.")
                    continue

            # Compute similarity scores for this batch
            # sim_scores = list(enumerate(cosine_similarities[idx]))
            sim_scores = list(enumerate(euclidean_similarities[idx]))
            all_sim_scores.extend([(batch_id, i[0], i[1]) for i in sim_scores])
            sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

            # Add similarity scores (excluding itself) to the global list
            sim_scores = sim_scores[1:11]  # Exclude itself and take top 10
            # all_sim_scores.extend([(batch_id, i[0], i[1]) for i in sim_scores])

    # Sort all similarity scores globally across batches
    all_sim_scores = sorted(all_sim_scores, key=lambda x: x[2], reverse=True)

    # Retrieve movie titles based on global similarity scores
    recommended_movies = pd.DataFrame(columns=['Title', 'Similarity', 'Batch'])
    for batch_id, movie_idx, similarity_score in all_sim_scores[:11]:  # Top 10 globally
        recommended_movies = pd.concat([recommended_movies, pd.DataFrame({
                'Title': [movie_titles_batches[batch_id].iloc[movie_idx]],
                'Similarity': [similarity_score],
                'Batch': [batch_id]
            })], ignore_index=True)
    # Set 'Title' as the index for dropping
    recommended_movies = recommended_movies.set_index('Title')

    # Remove any movie with 'title' in its name from the index
    recommended_movies = recommended_movies.drop(index=[title], errors='ignore').reset_index()

    return recommended_movies


example_title = 'Star Wars'
recommended_movies_all_batches = content_recommender_all_batches(example_title)

print(f"Recommendations for '{example_title}' across all batches:")
print(recommended_movies_all_batches)

Recommendations for 'Star Wars' across all batches:
                                          Title  Similarity    Batch
0       Star Wars Episode I  The Phantom Menace    1.000000  batch_3
1                      Star Wars The Clone Wars    1.000000  batch_5
2                  Star Wars Threads of Destiny    1.000000  batch_6
3               Inter Star Wars 2 The Last Jehi    1.000000  batch_7
4      Star Wars Episode VI  Return of the Jedi    0.477642  batch_2
5  Star Wars Episode V  The Empire Strikes Back    0.476333  batch_2
6    Star Wars Episode II  Attack of the Clones    0.468663  batch_3
7    Star Wars Episode III  Revenge of the Sith    0.466918  batch_3
8                                        Metron    0.459456  batch_7
9                                 Chistilishche    0.456579  batch_7


<ipython-input-12-05396e901533>:50: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  recommended_movies = pd.concat([recommended_movies, pd.DataFrame({


In [ ]:
 star_wars_recommendations = pd.DataFrame({
    'Title': ['Star Wars Episode I  The Phantom Menace', 'Star Wars The Clone Wars',
              'Star Wars Threads of Destiny', 'Inter Star Wars 2 The Last Jehi',
              'Star Wars Episode VI  Return of the Jedi', 'Star Wars Episode V  The Empire Strikes Back',
              'Star Wars Episode II  Attack of the Clones', 'Star Wars Episode III  Revenge of the Sith',
              'Metron', 'Chistilishche'],
    'Similarity': [1.0, 1.0, 1.0, 1.0, 0.477642, 0.476333, 0.468663, 0.466918, 0.459456, 0.456579],
    'Batch': ['batch_3', 'batch_5', 'batch_6', 'batch_7', 'batch_2', 'batch_2', 'batch_3', 'batch_3', 'batch_7', 'batch_7']
})

ground_truth = {}

ground_truth["Star Wars"] = {
    star_wars_recommendations.iloc[0]['Title']: 1,  # Relevant (Episode I)
    star_wars_recommendations.iloc[1]['Title']: 1,  # Relevant (The Clone Wars)
    star_wars_recommendations.iloc[2]['Title']: 1,  # Relevant (Threads of Destiny)
    star_wars_recommendations.iloc[3]['Title']: 0,  # Irrelevant (Inter Star Wars 2)
    star_wars_recommendations.iloc[4]['Title']: 1,  # Relevant (Episode VI)
    star_wars_recommendations.iloc[5]['Title']: 1,  # Relevant (Episode V)
    star_wars_recommendations.iloc[6]['Title']: 1,  # Relevant (Episode II)
    star_wars_recommendations.iloc[7]['Title']: 1,  # Relevant (Episode III)
    star_wars_recommendations.iloc[8]['Title']: 0,  # Irrelevant (Metron)
    star_wars_recommendations.iloc[9]['Title']: 0,  # Irrelevant (Chistilishche)
}

In [ ]:
def recall_at_k(recommendations, ground_truth, k):
    # Get top k recommendations
    top_k = recommendations.head(k)

    # Calculate relevant items in recommendations
    recommended_relevant = sum(ground_truth.get(title, 0) for title in top_k['Title'])

    # Calculate total relevant items in ground truth
    total_relevant = sum(ground_truth.values())

    # Avoid division by zero
    if total_relevant == 0:
        return 0.0

    return recommended_relevant / total_relevant

Recall@10: 100.00%


In [ ]:
def precision_at_k(recommendations, ground_truth, k):
    # Get top k recommendations
    top_k = recommendations.head(k)

    # Calculate relevant items in recommendations
    recommended_relevant = sum(ground_truth.get(title, 0) for title in top_k['Title'])

    # Avoid division by zero
    if k == 0:
        return 0.0

    return recommended_relevant / k

In [ ]:
def f1_score_at_k(recommendations, ground_truth, k):
    precision = precision_at_k(recommendations, ground_truth, k)
    recall = recall_at_k(recommendations, ground_truth, k)

    # Avoid division by zero
    if precision + recall == 0:
        return 0.0

    return 2 * (precision * recall) / (precision + recall)

# Calculate for all metrics at k=10
print(f"\nMetrics@10:")
print(f"Precision: {precision_at_k(star_wars_recommendations, ground_truth['Star Wars'], 10):.2%}")
print(f"Recall: {recall_at_k(star_wars_recommendations, ground_truth['Star Wars'], 10):.2%}")
print(f"F1-Score: {f1_score_at_k(star_wars_recommendations, ground_truth['Star Wars'], 10):.2%}")



Metrics@10:
Precision: 70.00%
Recall: 100.00%
F1-Score: 82.35%


Precision=
        All Retrieved Items
        --------------------------
        Relevant Retrieved Items

Recall=
        All Relevant Items in the Dataset
        ---------------------------------
        Relevant Retrieved Items

F1=2⋅ (Precision+Recall)/(Precision⋅Recall)

# **COSINE SIMILARITY**
Recommendations for 'Star Wars' across all batches:
                                           Title  Similarity    Batch
0                       Star Wars The Clone Wars    1.000000  batch_5
1                   Star Wars Threads of Destiny    1.000000  batch_6
2        Star Wars Episode I  The Phantom Menace    1.000000  batch_3
4                Inter Star Wars 2 The Last Jehi    1.000000  batch_7
5       Star Wars Episode VI  Return of the Jedi    0.402000  batch_2
6   Star Wars Episode V  The Empire Strikes Back    0.395690  batch_2
7     Star Wars Episode II  Attack of the Clones    0.357328  batch_3
8     Star Wars Episode III  Revenge of the Sith    0.348254  batch_3
9                                         Metron    0.307939  batch_7
10                                 Chistilishche    0.291712  batch_7

# **COSINE SIMILARITY** (without director and writer)
Recommendations for 'Star Wars' across all batches:
                                           Title  Similarity    Batch
0                   Star Wars Threads of Destiny    1.000000  batch_6
1                       Star Wars The Clone Wars    1.000000  batch_5
3                Inter Star Wars 2 The Last Jehi    1.000000  batch_7
4        Star Wars Episode I  The Phantom Menace    1.000000  batch_3
5   Star Wars Episode V  The Empire Strikes Back    0.429086  batch_2
6       Star Wars Episode VI  Return of the Jedi    0.388306  batch_2
7     Star Wars Episode II  Attack of the Clones    0.291019  batch_3
8     Star Wars Episode III  Revenge of the Sith    0.285369  batch_3
9                                         Metron    0.265297  batch_7
10                                       Krapiva    0.126577  batch_7

# **COSINE SIMILARITY** (only original_title and description)
Recommendations for 'Star Wars' across all batches:
                                            Title  Similarity    Batch
0       Star Wars: Episode I - The Phantom Menace    1.000000  batch_3
1                       Star Wars: The Clone Wars    1.000000  batch_5
2                   Star Wars: Threads of Destiny    1.000000  batch_6
3                Inter Star Wars 2. The Last Jehi    1.000000  batch_7
4  Star Wars: Episode V - The Empire Strikes Back    0.331028  batch_2
5      Star Wars: Episode VI - Return of the Jedi    0.302516  batch_2
6    Star Wars: Episode III - Revenge of the Sith    0.271020  batch_3
7                                      Spaceballs    0.223570  batch_2
8                                         Ore ore    0.218472  batch_6
9                                       SüperTürk    0.194185  batch_6

#**EUCLIDEAN DISTANCE**
Recommendations for 'Star Wars' across all batches:
                                           Title  Similarity    Batch
1        Star Wars Episode I  The Phantom Menace    1.000000  batch_3
2                       Star Wars The Clone Wars    1.000000  batch_5
3                   Star Wars Threads of Destiny    1.000000  batch_6
4                Inter Star Wars 2 The Last Jehi    1.000000  batch_7
5       Star Wars Episode VI  Return of the Jedi    0.477642  batch_2
6   Star Wars Episode V  The Empire Strikes Back    0.476333  batch_2
7     Star Wars Episode II  Attack of the Clones    0.468663  batch_3
8     Star Wars Episode III  Revenge of the Sith    0.466918  batch_3
9                                         Metron    0.459456  batch_7
10                                 Chistilishche    0.456579  batch_7

#**EUCLIDEAN DISTANCE** (without director and writer)
Recommendations for 'Star Wars' across all batches:
                                           Title  Similarity    Batch
1        Star Wars Episode I  The Phantom Menace    1.000000  batch_3
2                       Star Wars The Clone Wars    1.000000  batch_5
3                   Star Wars Threads of Destiny    1.000000  batch_6
4                Inter Star Wars 2 The Last Jehi    1.000000  batch_7
5   Star Wars Episode V  The Empire Strikes Back    0.483427  batch_2
6       Star Wars Episode VI  Return of the Jedi    0.474818  batch_2
7     Star Wars Episode II  Attack of the Clones    0.456458  batch_3
8     Star Wars Episode III  Revenge of the Sith    0.455474  batch_3
9                                         Metron    0.452041  batch_7
10                                       Krapiva    0.430722  batch_7

# **EUCLIDEAN DISTANCE** (only original_title and description)
Recommendations for 'Star Wars' across all batches:
                                            Title  Similarity    Batch
0       Star Wars: Episode I - The Phantom Menace    1.000000  batch_3
1                       Star Wars: The Clone Wars    1.000000  batch_5
2                   Star Wars: Threads of Destiny    1.000000  batch_6
3                Inter Star Wars 2. The Last Jehi    1.000000  batch_7
4  Star Wars: Episode V - The Empire Strikes Back    0.463672  batch_2
5      Star Wars: Episode VI - Return of the Jedi    0.458487  batch_2
6    Star Wars: Episode III - Revenge of the Sith    0.453009  batch_3
7                                      Spaceballs    0.445209  batch_2
8                                         Ore ore    0.444401  batch_6
9                                       SüperTürk    0.440626  batch_6

**GROUND TRUTH DATASET**
# Star Wars (Dataset order)


1.   Star Wars
1.   Star Wars: Episode V - The Empire Strikes Back
1.   Star Wars: Episode VI - Return of the Jedi
1.   Star Wars: Episode I - The Phantom Menace
2.   Star Wars: Episode II - Attack of the Clones
2.   Star Wars: Episode III - Revenge of the Sith
2.   Star Wars: The Clone Wars
2.   Star Wars: Threads of Destiny
1.   Star Wars: Episode VII - The Force Awakens
2.   Star Wars: Episode VIII - The Last Jedi
1.   Star Wars: Episode IX - The Rise of Skywalker
2.   Rogue One (Original Title is different from title)
1.   Solo: A Star Wars Story